# Prophage relatedness and host–phage association analysis

This notebook reproduces the prophage-relatedness network and the within-*L. plantarum* Mantel analysis described in the manuscript. The relatedness metric is based on reciprocal-best-hit (RBH) similarity in anisotropy-corrected ESM2 protein embeddings.

The analysis is descriptive/comparative and does not establish prophage transmission, contemporary phage circulation, or active CRISPR interference.


## 1. Setup

In [ ]:
!pip install -q biopython dendropy scikit-bio networkx
import numpy as np, pandas as pd, os


## 2. Load inputs (reuse in-memory objects if available, else upload)

In [ ]:
from google.colab import files

def ensure_file(varname_hint, filename):
    if os.path.exists(filename):
        return filename
    print(f"Please upload {filename}:")
    up = files.upload()
    for f in up:
        if f != filename:
            os.rename(f, filename)
    return filename

# Master tables (small, always re-upload to be safe)
master_path = ensure_file("master", "MASTER_prophage_validation_table.csv")
func_path   = ensure_file("func", "esm2_putative_function_calls.csv")

master = pd.read_csv(master_path)
func_df = pd.read_csv(func_path)
print(f"master: {master.shape}, func_df: {func_df.shape}")

# Embeddings -- reuse the in-memory `embeddings` dict from the earlier notebook if present in this session
if "embeddings" in globals() and len(globals()["embeddings"]) > 0:
    print(f"Using in-memory embeddings dict ({len(embeddings)} proteins)")
else:
    emb_path = ensure_file("embeddings", "esm2_embeddings.npz")
    _npz = np.load(emb_path, allow_pickle=True)
    embeddings = {k: _npz[k] for k in _npz.files}
    print(f"Loaded {len(embeddings)} embeddings from {emb_path}")

# Tree file
tree_path = ensure_file("tree", "core_gene_alignment_filtered_aln.treefile")
print("Tree file ready:", tree_path)


## 3. Prepare anisotropy-corrected protein embeddings

The prophage relatedness metric used in the manuscript is based on reciprocal-best-hit (RBH) protein similarity, not the cosine similarity of mean prophage profiles. ESM2 vectors are therefore mean-centered first, then used to compute reciprocal best protein matches between each pair of geNomad-confirmed prophages.

In [ ]:
from numpy.linalg import norm

confirmed = master.loc[master["genomad_call"].eq("virus"), "header"].tolist()
all_vecs = np.vstack(list(embeddings.values()))
global_mean = all_vecs.mean(axis=0)
centered = {k: v - global_mean for k, v in embeddings.items()}

by_prophage = {h: [k for k in embeddings if k.startswith(h + "__")] for h in confirmed}
by_prophage = {h: ids for h, ids in by_prophage.items() if ids}
print(f"Confirmed viral prophages with usable protein embeddings: {len(by_prophage)}")

def rbh_similarity(h1, h2):
    p1, p2 = by_prophage[h1], by_prophage[h2]
    V1 = np.vstack([centered[p] for p in p1]); V1 /= (norm(V1, axis=1, keepdims=True) + 1e-9)
    V2 = np.vstack([centered[p] for p in p2]); V2 /= (norm(V2, axis=1, keepdims=True) + 1e-9)
    S = V1 @ V2.T
    return float(np.concatenate([S.max(axis=1), S.max(axis=0)]).mean())


## 4. Cross-species prophage-relatedness network

For every pair of confirmed prophages, calculate the mean reciprocal-best-hit protein similarity. Connect only the **upper 5%** of pairwise similarity scores, matching the method reported in the manuscript.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

ids = list(by_prophage.keys())
pair_scores = {}
for i, h1 in enumerate(ids):
    for h2 in ids[i+1:]:
        pair_scores[(h1, h2)] = rbh_similarity(h1, h2)

scores = np.array(list(pair_scores.values()))
threshold = np.quantile(scores, 0.95)
print(f"RBH similarity 95th-percentile threshold: {threshold:.4f}")

G = nx.Graph()
acc_of = master.set_index("header")["accession"].to_dict()
species_map = {
    'GCA_000392485.2':'L. plantarum','GCF_000008065.1':'L. johnsonii','GCF_000010145.1':'L. fermentum',
    'GCF_000011985.1':'L. acidophilus','GCF_000014425.1':'L. gasseri','GCF_000014525.1':'L. paracasei',
    'GCF_000016825.1':'L. reuteri','GCF_000019245.4':'L. paracasei','GCF_000023085.1':'L. plantarum',
    'GCF_000026505.1':'L. rhamnosus','GCF_000148815.2':'L. plantarum','GCF_000165775.1':'L. helveticus',
    'GCF_000203855.3':'L. plantarum','GCF_000248095.2':'L. mucosae','GCF_000338115.2':'L. plantarum',
    'GCF_000389675.2':'L. acidophilus','GCF_000412205.1':'L. plantarum','GCF_001704335.1':'L. plantarum',
    'GCF_014131735.1':'L. plantarum','GCF_041888805.1':'L. reuteri',
}
for h in ids:
    G.add_node(h, species=species_map.get(acc_of.get(h), "unknown"))
for (h1, h2), score in pair_scores.items():
    if score >= threshold:
        G.add_edge(h1, h2, weight=float(score))

cross_species_edges = [(u, v) for u, v in G.edges() if G.nodes[u]["species"] != G.nodes[v]["species"]]
print(f"Total edges: {G.number_of_edges()}")
print(f"Cross-species edges: {len(cross_species_edges)}")
for u, v in cross_species_edges:
    print(f"  {u} ({G.nodes[u]['species']}) <-> {v} ({G.nodes[v]['species']}) sim={G[u][v]['weight']:.3f}")


In [ ]:
species_list = sorted(set(nx.get_node_attributes(G, "species").values()))
cmap = plt.cm.tab20
colors = {s: cmap(i / max(len(species_list)-1, 1)) for i, s in enumerate(species_list)}
node_colors = [colors[G.nodes[n]["species"]] for n in G.nodes()]

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G, seed=42, k=0.6)
nx.draw_networkx_edges(G, pos, alpha=0.4)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=180)
handles = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=colors[s], label=s, markersize=9)
           for s in species_list]
plt.legend(handles=handles, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.title("Prophage relatedness network (geNomad-confirmed regions)\nRBH ESM2 similarity; upper 5% of pairwise scores")
plt.axis("off")
plt.tight_layout()
plt.savefig("/content/prophage_sharing_network.png", dpi=200, bbox_inches="tight")
plt.show()


## 5. Host–phage association test within *L. plantarum*

The test is restricted to the eight *L. plantarum* strains and uses host patristic distances from the IQ-TREE core-genome phylogeny and prophage-relatedness distances derived from the RBH ESM2 similarity metric. The non-significant result is interpreted as inconclusive given the small number of strains.


In [ ]:
import dendropy
from skbio.stats.distance import mantel, DistanceMatrix

PLANTARUM_ACCESSIONS = {
    'GCA_000392485.2': 'P-8', 'GCF_000023085.1': 'JDM1', 'GCF_000148815.2': 'ST-III',
    'GCF_000203855.3': 'WCFS1', 'GCF_000338115.2': 'ZJ316', 'GCF_000412205.1': '16',
    'GCF_001704335.1': 'DF', 'GCF_014131735.1': 'DSM20174',
}

tree = dendropy.Tree.get(path=tree_path, schema="newick")
pdm = tree.phylogenetic_distance_matrix()
taxon_by_label = {t.label: t for t in tree.taxon_namespace}
print("Tree tip labels found:", list(taxon_by_label.keys()))

def normalize(s):
    return s.lower().replace("_", "").replace("-", "").replace(" ", "")

def match_taxon(strain_name):
    target = normalize(strain_name)
    # exact match first (avoids short names like "16" matching inside "ZJ316")
    for label, t in taxon_by_label.items():
        if normalize(label) == target:
            return t
    # fallback: substring match only for longer, less ambiguous names
    if len(target) >= 4:
        for label, t in taxon_by_label.items():
            if target in normalize(label) or normalize(label) in target:
                return t
    return None

strain_taxon = {acc: match_taxon(name) for acc, name in PLANTARUM_ACCESSIONS.items()}
strain_taxon = {k: v for k, v in strain_taxon.items() if v is not None}
print(f"Matched {len(strain_taxon)} / {len(PLANTARUM_ACCESSIONS)} L. plantarum strains to tree tips")
print({k: v.label for k, v in strain_taxon.items()})


In [ ]:
# Host (patristic) distance matrix, restricted to the eight *L. plantarum* strains
usable_accs = [a for a in strain_taxon if
               any(master.loc[master["header"]==h, "accession"].iloc[0] == a for h in confirmed_profiles)]
print(f"Usable L. plantarum strains: {len(usable_accs)}")

host_dist = pd.DataFrame(index=usable_accs, columns=usable_accs, dtype=float)
for a in usable_accs:
    for b in usable_accs:
        host_dist.loc[a, b] = pdm.patristic_distance(strain_taxon[a], strain_taxon[b])

# Map confirmed prophage regions to their host accessions.
acc_to_headers = {a: [h for h in confirmed_profiles
                       if master.loc[master["header"]==h, "accession"].iloc[0] == a]
                  for a in usable_accs}

print("Host distance matrix prepared from the IQ-TREE core-genome phylogeny.")


In [ ]:
host_dm = DistanceMatrix(host_dist.values, ids=usable_accs)
# Convert RBH similarity to a prophage-relatedness distance.
phage_dist = pd.DataFrame(index=usable_accs, columns=usable_accs, dtype=float)
for a in usable_accs:
    for b in usable_accs:
        if a == b:
            phage_dist.loc[a, b] = 0.0
        else:
            ha = acc_to_headers[a]; hb = acc_to_headers[b]
            sims = [rbh_similarity(h1, h2) for h1 in ha for h2 in hb]
            phage_dist.loc[a, b] = 1.0 - float(np.mean(sims))

phage_dm = DistanceMatrix(phage_dist.values, ids=usable_accs)
r, p_value, n = mantel(host_dm, phage_dm, method="pearson", permutations=999)
r_s, p_s, _ = mantel(host_dm, phage_dm, method="spearman", permutations=999)
print("Mantel test (host phylogenetic distance vs. prophage relatedness distance)")
print(f"  N strains compared: {n}")
print(f"  Pearson r = {r:.3f}, p = {p_value:.4f}")
print(f"  Spearman r = {r_s:.3f}, p = {p_s:.4f}")
print("  Manuscript-reported result: Pearson r=0.24, p=0.09; Spearman r=-0.05, p=0.74")
print("  Interpretation: non-significant and inconclusive at n=8; this does not establish absence of co-diversification or prove horizontal acquisition.")


## 6. Interpretation

The relatedness network identifies sequence-level similarity among prophage-associated regions. Cross-species connections should be interpreted cautiously: sequence relatedness alone cannot distinguish recent horizontal transfer, common ancestry, or independent acquisition from a shared environmental phage pool.
